# 01 - Geleneksel CSP + LDA

**Amac:** Guclu, yorumlanabilir, dusuk-hesap maliyetli bir MI temel hatti.

**Protokol (adil, birincil):** train uzerinde fit, validation ile secim, test'te **bir kez** degerlendirme. Birincil karsilastirmada train+val yeniden fit YOK.

**Beklenen ciktilar:** `results/csp_lda/` altinda metrikler, secilen hiperparametreler, test tahminleri, modeller, sekiller, toplu metrikler.

In [ ]:
# --- Ortak baslangic: proje koku kesfi ve ice aktarmalar ---
import sys, json, warnings
from pathlib import Path

def _find_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / "pyproject.toml").exists() and (d / "src" / "cho2017_benchmark").exists():
            return d
    return p

ROOT = _find_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cho2017_benchmark import paths
from cho2017_benchmark.config import resolve_config, dump_resolved_config
from cho2017_benchmark.reproducibility import set_seed, save_environment

set_seed(42)
print("Proje koku:", paths.PROJECT_ROOT)

In [ ]:
QUICK_MODE = False  # Yalnizca hata ayiklama icindir; bilimsel sonuc uretmez.
cfg = resolve_config("csp_lda", quick_mode=QUICK_MODE)
if cfg.quick_mode:
    print("UYARI: QUICK_MODE sonuclari nihai bilimsel sonuc olarak kullanilamaz.")
    print("QUICK_MODE denekleri:", cfg.active_subjects())
print("Aktif denek sayisi:", len(cfg.active_subjects()), "| Cihaz politikasi:", cfg.device)
print("Siniflar:", cfg.class_names, "| Ornekleme:", cfg.srate_in, "->", cfg.srate_out, "Hz")

In [ ]:
# Gerekli veri dosyalarini ve manifestolari dogrula
from cho2017_benchmark.data.inspect_mat import load_layout_resolution
resolution = load_layout_resolution()
print("Layout dogrulandi mi:", resolution["validated"], "| yonelim:", resolution["orientation"])

manifest_path = paths.manifests_dir() / "split_manifest.csv"
assert manifest_path.exists(), f"Split manifestosu yok: {manifest_path}. Once create_splits.py calistirin."
split_manifest = pd.read_csv(manifest_path)
print("Split manifestosu:", split_manifest.shape, "| denek:", split_manifest.subject_id.nunique())

from cho2017_benchmark.data.preprocessing import PreprocessingConfig
from cho2017_benchmark.reproducibility import preprocessing_hash, split_manifest_hash
PP = PreprocessingConfig.from_config(cfg)
PREPROC_HASH = preprocessing_hash(PP.hashable())
SPLIT_HASH = split_manifest_hash(split_manifest)
print("preprocessing_hash:", PREPROC_HASH, "| split_manifest_hash:", SPLIT_HASH)

## Denek bazli CSP+LDA dongusu
Her denek icin `run_subject_csp` cagrilir (src icindeki surucu).

In [ ]:
from cho2017_benchmark.models.csp_lda import run_subject_csp
from cho2017_benchmark.data import prepare
from cho2017_benchmark.data.datasets import subject_split_indices, subject_split_meta
import joblib, logging

MODEL = "csp_lda"
base = paths.results_dir(MODEL)
for sub in ["tables", "predictions", "metrics", "figures", "models", "configs", "logs"]:
    (base / sub).mkdir(parents=True, exist_ok=True)
logging.basicConfig(filename=base / "logs" / "experiment.log", level=logging.INFO, force=True)

all_metrics, all_preds, all_selected, all_cands, failures = [], [], [], [], []
for idx in cfg.active_subjects():
    sid = f"s{idx:02d}"
    try:
        rec = prepare.load_processed(idx)
        indices = subject_split_indices(rec, split_manifest, sid)
        meta = subject_split_meta(rec, indices)
        sm = split_manifest[split_manifest.subject_id == sid]
        split_method = sm["split_method"].iloc[0] if len(sm) else "unknown"
        model_path = str(base / "models" / f"{sid}.joblib")
        out = run_subject_csp(cfg, rec["X_eeg"], rec["y"], indices, subject_id=sid,
                              split_method=split_method, split_manifest_hash=SPLIT_HASH,
                              preprocessing_hash=PREPROC_HASH, split_meta_test=meta["test"],
                              model_path=model_path)
        joblib.dump(out["model"], model_path)
        all_metrics.append(out["metrics"]); all_preds.append(out["predictions"])
        all_selected.append(out["selected"]); all_cands.extend(out["candidates"])
        logging.info("%s ok acc=%.3f", sid, out["metrics"]["accuracy"])
    except Exception as e:  # basarisizliklar makinece-okunur kaydedilir
        from cho2017_benchmark.evaluation.metrics import compute_subject_metrics
        failures.append({"subject_id": sid, "error": str(e)})
        all_metrics.append(compute_subject_metrics(None, None, None, model_name=MODEL, subject_id=sid,
            split_method="", n_train=0, n_validation=0, n_test=0, seed=cfg.seed,
            split_manifest_hash=SPLIT_HASH, preprocessing_hash=PREPROC_HASH,
            status="failed", error_message=str(e)))
        logging.exception("%s basarisiz", sid)
print("Tamamlanan:", len(all_metrics), "| basarisiz:", len(failures))

## Ciktilari kaydet (sema dogrulamali)

In [ ]:
from cho2017_benchmark.reporting.result_writer import (
    write_subject_metrics, write_predictions, write_selected_hyperparameters, write_aggregate_metrics)
from cho2017_benchmark.evaluation.statistics import aggregate_seeds_per_subject

write_subject_metrics(all_metrics, base / "tables" / "subject_metrics.csv")
write_predictions(all_preds, base / "predictions" / "test_predictions.parquet")
write_selected_hyperparameters(all_selected, base / "tables" / "selected_hyperparameters.csv")
metrics_df = pd.DataFrame(all_metrics)
per_subject = aggregate_seeds_per_subject(metrics_df)
write_aggregate_metrics(per_subject, base / "metrics" / "aggregate_metrics.json")
dump_resolved_config(cfg, base / "configs" / "resolved_config.yaml")
save_environment(base)
(base / "logs" / "failures.json").write_text(json.dumps(failures, indent=2), encoding="utf-8")
print("CSP+LDA ciktilari yazildi. Ortalama dogruluk:",
      round(float(per_subject["accuracy"].mean()), 3) if len(per_subject) else float("nan"))

## Yorumlanabilirlik + ozet sekiller
CSP desenleri (topomap) ve denek dogruluk dagilimlari.

In [ ]:
from cho2017_benchmark.evaluation import plots
ok = metrics_df[metrics_df["status"] == "ok"]
if len(ok):
    plots.save_fig(plots.subject_accuracy_sorted(ok), base / "figures" / "subject_accuracy_sorted.png")
    plots.save_fig(plots.subject_accuracy_distribution(ok), base / "figures" / "subject_accuracy_distribution.png")
    cm = np.array([[int(ok["tn"].sum()), int(ok["fp"].sum())], [int(ok["fn"].sum()), int(ok["tp"].sum())]])
    plots.save_fig(plots.confusion_matrix_plot(cm, cfg.class_names, title="CSP+LDA toplu"),
                   base / "figures" / "aggregate_confusion_matrix.png")
plt.show()

## Limitations
- CSP guclu bir temel hattir; bazi denekler icin derin modelleri gecebilir.
- Cevrimdisi sonuc; cevrimici kullanilabilirlik garantisi degildir.

## Generated Files
`results/csp_lda/tables/*.csv`, `predictions/test_predictions.parquet`, `metrics/aggregate_metrics.json`, `models/<sid>.joblib`, `figures/*.png`, `configs/resolved_config.yaml`.

In [ ]:
expected = [base / "tables" / "subject_metrics.csv",
            base / "predictions" / "test_predictions.parquet",
            base / "metrics" / "aggregate_metrics.json"]
missing = [str(p) for p in expected if not p.exists()]
assert not missing, f"Beklenen CSP+LDA ciktilari eksik: {missing}"
print("Tum beklenen CSP+LDA ciktilari mevcut.")